# Chapter 12 - Dynamic Aperture

Dynamic aperture (DA) is measure of the stable phase space in an accelerator over a specified number of turns. Particles outside of the DA will be driven to large amplitudes where they will be lost. In the following sections, strategies for increasing a ring's DA will be shown in a sample ring.

![Dynamic aperture ray construction](assets/chapter12_dynamic_aperture_rays.svg)

**Figure 1:** The calculation of a dynamic aperture curve in the $x-y$ plane at a given initial $p_z$ value involves calculating aperture curve points (blue dots) along a set of "rays" (dashed lines). The line segments between points is simply for visualization purposes and does not appear in graphs of real data.


In [ ]:
# SciBmad builds and tracks the lattice; CairoMakie makes the figures.
using SciBmad
using CairoMakie

# LinearAlgebra is used for eigenvalues and response-matrix solves.
using LinearAlgebra
using Printf


## 12.1 Calculating Dynamic Aperture

Points along rays are tested to determine the dynamic-aperture perimeter. All rays begin at the reference orbit. Along each ray, particles with different starting $(x,y)$ positions are tracked to find the boundary between stable and unstable motion.

For this compact SciBmad example, a particle is classified as stable when:

- every tracked coordinate remains finite;
- the transverse coordinates remain inside a generous $50\ \mathrm{mm}$ numerical guard aperture; and
- the particle completes the requested number of turns.

The guard aperture is not the dynamic aperture itself. It prevents a clearly diverging trajectory from consuming unnecessary tracking time.


In [ ]:
function build_chapter12_ring(;
    n_cells=16,
    chromatic_strength=300.0,
    harmonic_strength=0.0,
    chromatic_sf_strength=chromatic_strength,
    chromatic_sd_strength=-chromatic_strength,
    harmonic_sf_strength=harmonic_strength,
    harmonic_sd_strength=-harmonic_strength,
)
    elements = SciBmad.LineElement[]

    for i in 1:n_cells
        # Odd cells model the chromatic sextupole families. Even cells model
        # the harmonic families that are turned on in Section 12.2.
        sf = isodd(i) ? chromatic_sf_strength : harmonic_sf_strength
        sd = isodd(i) ? chromatic_sd_strength : harmonic_sd_strength

        # Each compact cell is FODO-like: focus, sextupole, bend, sextupole,
        # defocus. This is a small teaching ring, not the full ESR lattice.
        append!(elements, [
            Quadrupole(name="QF_$i", L=0.25, Kn1=0.8),
            Drift(name="D1A_$i", L=0.20),
            Sextupole(name="SF_$i", L=0.10, Kn2=sf),
            Drift(name="D1B_$i", L=0.20),
            SBend(name="B_$i", L=1.0, angle=2pi / n_cells),
            Drift(name="D2A_$i", L=0.20),
            Sextupole(name="SD_$i", L=0.10, Kn2=sd),
            Drift(name="D2B_$i", L=0.20),
            Quadrupole(name="QD_$i", L=0.25, Kn1=-0.8),
            Drift(name="D3_$i", L=0.40),
        ])
    end

    return Beamline(elements; species_ref=Species("electron"), E_ref=3e9)
end

# The first ring has only the chromatic families. The second includes the
# harmonic families used for the first dynamic-aperture plot.
ring_chromatic = build_chapter12_ring()
ring_with_harmonic_sextupoles = build_chapter12_ring(
    chromatic_strength=300.0,
    harmonic_strength=300.0,
)

@printf("Tracked elements: %d
", length(ring_chromatic.line))
@printf("Circumference: %.3f m
", sum(element.L for element in ring_chromatic.line))


The aperture point on one ray is found in two stages:

1. scan outward in fixed radial steps until the first unstable launch point is found;
2. refine the last stable/first unstable bracket by bisection.

This is a finite-turn numerical definition of DA. Increasing the tracking time can reveal slow losses and therefore reduce the measured aperture.


In [ ]:
function reference_bunch(ring, coordinates)
    coordinates = reshape(coordinates, 1, 6)

    # Recent BeamTracking versions use p_over_q_ref; older versions used R_ref.
    # Keeping both paths lets the notebook run across nearby SciBmad releases.
    if hasproperty(ring, :p_over_q_ref)
        return Bunch(
            coordinates;
            species=ring.species_ref,
            p_over_q_ref=ring.p_over_q_ref,
        )
    elseif hasproperty(ring, :R_ref)
        return Bunch(
            coordinates;
            species=ring.species_ref,
            R_ref=ring.R_ref,
        )
    else
        return Bunch(coordinates; species=ring.species_ref)
    end
end

function stable_motion(
    ring,
    x,
    y;
    pz=0.0,
    n_turns=80,
    guard_aperture=0.05,
)
    # Launch one particle at the requested transverse coordinate and momentum offset.
    bunch = reference_bunch(ring, [x, 0.0, y, 0.0, 0.0, pz])

    try
        for _ in 1:n_turns
            track!(bunch, ring)
            coordinates = Array(bunch.coords.v)

            # Treat NaNs/Infs or very large transverse excursions as particle loss.
            all(isfinite, coordinates) || return false
            maximum(abs, coordinates[:, [1, 3]]) < guard_aperture || return false
        end
    catch
        # Failed tracking usually means the launch point has become unstable.
        return false
    end

    return true
end

function ray_boundary(
    ring,
    theta;
    pz=0.0,
    n_turns=80,
    r_max=0.03,
    radial_step=0.001,
    refinements=6,
)
    last_stable_radius = 0.0

    # First scan outward along one ray until the first unstable point is found.
    for radius in radial_step:radial_step:r_max
        x = radius * cos(theta)
        y = radius * sin(theta)

        if !stable_motion(ring, x, y; pz=pz, n_turns=n_turns)
            low = last_stable_radius
            high = radius

            # Refine the stable/unstable bracket by bisection.
            for _ in 1:refinements
                middle = (low + high) / 2
                middle_is_stable = stable_motion(
                    ring,
                    middle * cos(theta),
                    middle * sin(theta);
                    pz=pz,
                    n_turns=n_turns,
                )
                middle_is_stable ? (low = middle) : (high = middle)
            end

            return low
        end

        last_stable_radius = radius
    end

    # If the whole ray is stable up to r_max, report the scan limit.
    return r_max
end

function dynamic_aperture(
    ring;
    pz_values=[0.0],
    n_angles=9,
    n_turns=80,
    r_max=0.03,
)
    angles = range(0, pi; length=n_angles)
    curves = NamedTuple[]

    # Repeat the ray search for each momentum offset.
    for pz in pz_values
        radii = [
            ray_boundary(
                ring,
                theta;
                pz=pz,
                n_turns=n_turns,
                r_max=r_max,
            )
            for theta in angles
        ]

        push!(curves, (
            pz=pz,
            x=radii .* cos.(angles),
            y=radii .* sin.(angles),
            radii=radii,
        ))
    end

    return curves
end


The original example evaluates several momentum offsets. For this compact teaching ring, we scan $p_z=0$ through $p_z=0.007$, corresponding to $\delta=0.0\%$ through $0.7\%$ in $0.1\%$ steps. The plot is normalized by nominal beam sizes, $\sigma_x=\sigma_y=1\ \mathrm{mm}$, matching the style of the SciBmad dynamic-aperture example. This demonstration tracks 80 turns and uses 31 rays. Increase `n_turns` and `n_angles` for a production study.


In [ ]:
# Scan momentum offsets in 0.1% steps.
pz_values = collect(0.000:0.001:0.007)

# Dynamic aperture is evaluated after turning on the harmonic sextupoles.
corrected_da = dynamic_aperture(
    ring_with_harmonic_sextupoles;
    pz_values=pz_values,
    n_angles=31,
    n_turns=80,
    r_max=0.03,
)

# Print a compact numerical summary for each curve.
for curve in corrected_da
    @printf(
        "pz = %.4f: mean boundary radius = %.2f mm
",
        curve.pz,
        1e3 * sum(curve.radii) / length(curve.radii),
    )
end


In [ ]:
function plot_dynamic_aperture(
    curves;
    title="Dynamic Aperture",
    sigma_x=1.0e-3,
    sigma_y=1.0e-3,
)
    figure = Figure(size=(820, 600))
    axis = Axis(
        figure[1, 1];
        xlabel="x / sigma_x",
        ylabel="y / sigma_y",
        title=title,
        aspect=DataAspect(),
    )

    x_limit = 0.0
    y_limit = 0.0

    for curve in curves
        # Normalize the physical aperture coordinates by nominal beam sizes.
        x_norm = curve.x ./ sigma_x
        y_norm = curve.y ./ sigma_y
        x_limit = max(x_limit, maximum(abs, x_norm))
        y_limit = max(y_limit, maximum(y_norm))
        label = @sprintf("delta = %.1f%%", 100 * curve.pz)
        lines!(axis, x_norm, y_norm; linewidth=2.8, label=label)
    end

    # Use symmetric x limits so x=0 appears at the visual center.
    x_limit = ceil(x_limit)
    y_limit = ceil(y_limit)
    xlims!(axis, -x_limit, x_limit)
    ylims!(axis, 0, y_limit)

    axislegend(axis; position=:rt)
    return figure
end

chromatic_da_figure = plot_dynamic_aperture(
    corrected_da;
    title="Dynamic Aperture with Chromatic and Harmonic Sextupoles",
)
save("assets/chapter12_dynamic_aperture.png", chromatic_da_figure)
chromatic_da_figure


The resulting curves are the SciBmad equivalent of Figure 24 in the original tutorial, plotted in the normalized style used by the SciBmad dynamic-aperture example. Each colored curve represents one momentum offset. The calculated vertices are connected for visualization only, so small jagged features are expected when the scan uses a finite number of rays.

The chromatic-only ring can have a much rougher aperture. Section 12.2 compares it with the corrected ring to show why harmonic sextupoles are useful.


## 12.2 Example: Adding Sextupoles

A common element used for DA optimization are sextupoles. "Harmonic" sextupoles, that is, sextupoles that minimize resonance driving modes, can be used to increase the transverse aperture. Sextupoles can be added to the lattice by following the steps below.

The original Bmad example performs the following structural changes:

1. define chromatic sextupoles in dispersive arc regions;
2. define harmonic sextupoles in straight sections;
3. split drifts to make room for the new elements;
4. modify the forward and reverse FODO cells;
5. rebuild the ring from the altered cells.

Our compact ring already splits the drifts around every sextupole location. Odd cells contain the chromatic families `SF` and `SD`. In the first calculation the even-cell harmonic families were present but had zero strength. We now turn those families on without changing the linear quadrupole lattice.


In [ ]:
# Rebuild the same linear lattice, now with nonzero harmonic sextupoles.
ring_with_harmonic_sextupoles = build_chapter12_ring(
    chromatic_strength=300.0,
    harmonic_strength=300.0,
)

@printf("Chromatic SF/SD strength magnitude: %.1f
", 300.0)
@printf("Harmonic  SF/SD strength magnitude: %.1f
", 300.0)


To see whether the added harmonic family helps, we repeat the on-momentum ray search. This is the essential DA optimization loop: change sextupole settings, recalculate the aperture, and retain changes that improve the desired stability measure.


In [ ]:
# Compare on-momentum dynamic aperture before and after the harmonic family is added.
chromatic_only_da = dynamic_aperture(
    ring_chromatic;
    pz_values=[0.0],
    n_angles=31,
    n_turns=80,
    r_max=0.03,
)

harmonic_da = dynamic_aperture(
    ring_with_harmonic_sextupoles;
    pz_values=[0.0],
    n_angles=31,
    n_turns=80,
    r_max=0.03,
)

before_mean = 1e3 * sum(chromatic_only_da[1].radii) / length(chromatic_only_da[1].radii)
after_mean = 1e3 * sum(harmonic_da[1].radii) / length(harmonic_da[1].radii)

@printf("Mean on-momentum boundary before harmonic sextupoles: %.2f mm
", before_mean)
@printf("Mean on-momentum boundary after harmonic sextupoles:  %.2f mm
", after_mean)


In [ ]:
comparison_figure = Figure(size=(820, 600))
comparison_axis = Axis(
    comparison_figure[1, 1];
    xlabel="x / sigma_x",
    ylabel="y / sigma_y",
    title="Effect of Adding Harmonic Sextupoles",
    aspect=DataAspect(),
)

sigma_x = 1.0e-3
sigma_y = 1.0e-3

# Plot the chromatic-only and chromatic-plus-harmonic boundaries together.
lines!(
    comparison_axis,
    chromatic_only_da[1].x ./ sigma_x,
    chromatic_only_da[1].y ./ sigma_y;
    linewidth=2.8,
    label="chromatic families only",
)

lines!(
    comparison_axis,
    harmonic_da[1].x ./ sigma_x,
    harmonic_da[1].y ./ sigma_y;
    linewidth=2.8,
    label="chromatic + harmonic families",
)

# Match the plotting convention used above: symmetric x limits and y >= 0.
comparison_x_limit = ceil(max(
    maximum(abs, chromatic_only_da[1].x ./ sigma_x),
    maximum(abs, harmonic_da[1].x ./ sigma_x),
))
comparison_y_limit = ceil(max(
    maximum(chromatic_only_da[1].y ./ sigma_y),
    maximum(harmonic_da[1].y ./ sigma_y),
))
xlims!(comparison_axis, -comparison_x_limit, comparison_x_limit)
ylims!(comparison_axis, 0, comparison_y_limit)

axislegend(comparison_axis; position=:rt)
save("assets/chapter12_adding_sextupoles.png", comparison_figure)
comparison_figure


For this demonstration setting, the harmonic family increases the mean on-momentum boundary radius. This result is not a general rule that stronger sextupoles always improve DA. Sextupole strengths and phase relationships must be optimized, and the final settings must be checked over the required momentum range and tracking time.


## 12.3 Example: Chromaticity Correction

The original example lattice in bmad for Section 12.3 introduces two overlay knobs for the arc chromatic sextupoles:

```bmad
OSF: overlay = {SF%_*[k2]: x}, var = {x}
OSD: overlay = {SD%_*[k2]: x}, var = {x}
```

The matching `tao.init` file varies those two knobs and sets the targets `chrom.a = 1` and `chrom.b = 1`. 
In this SciBmad notebook, we keep the same idea: one knob changes all chromatic `SF` sextupoles, and the other changes all chromatic `SD` sextupoles. The full Bmad ESR lattice is not rebuilt here; instead, the same two-family correction is applied to the compact teaching ring used above.

In this optimization, the varied variables are the two sextupole-family knobs `OSF` and `OSD`, which set the common `k2` strengths of the arc `SF` and `SD` families. The optimized quantities are the two chromaticities, `chrom.a` and `chrom.b`, with target values 1 and 1.


In [28]:
function ring_with_os_knobs(os; harmonic_strength=0.0)
    # os[1] is the OSF knob; os[2] is the OSD knob.
    return build_chapter12_ring(
        chromatic_sf_strength=os[1],
        chromatic_sd_strength=os[2],
        harmonic_strength=harmonic_strength,
    )
end

function one_turn_transverse_coordinates(ring, u; pz=0.0)
    # Track one particle for one turn and keep only (x, px, y, py).
    bunch = reference_bunch(ring, [u[1], u[2], u[3], u[4], 0.0, pz])
    track!(bunch, ring)
    coordinates = Array(bunch.coords.v)
    return vec(coordinates[1, [1, 2, 3, 4]])
end

function one_turn_transverse_matrix(ring; pz=0.0, step=1e-7)
    matrix = zeros(4, 4)
    origin = zeros(4)

    for j in 1:4
        # Centered finite difference gives one column of the transverse map.
        perturbation = zeros(4)
        perturbation[j] = step
        matrix[:, j] = (
            one_turn_transverse_coordinates(ring, origin .+ perturbation; pz=pz) .-
            one_turn_transverse_coordinates(ring, origin .- perturbation; pz=pz)
        ) ./ (2step)
    end

    return matrix
end

function tunes_from_matrix(matrix)
    # Stable transverse motion appears as two complex-conjugate eigenvalue pairs.
    eigenvalues = eigvals(matrix)
    tunes = sort(mod.(angle.(eigenvalues[imag.(eigenvalues) .> 0]) ./ (2pi), 1.0))
    length(tunes) == 2 || error("Expected two stable transverse modes.")
    return tunes
end

function tunes_from_tracking(ring; pz=0.0)
    return tunes_from_matrix(one_turn_transverse_matrix(ring; pz=pz))
end

function chromaticity_from_tracking(os; harmonic_strength=0.0, dpz=1e-4)
    # Chromaticity is dQ/dp_z, approximated by a centered difference.
    ring = ring_with_os_knobs(os; harmonic_strength=harmonic_strength)
    return (
        tunes_from_tracking(ring; pz=dpz) .-
        tunes_from_tracking(ring; pz=-dpz)
    ) ./ (2dpz)
end

function chromaticity_response(os; harmonic_strength=0.0, knob_step=0.01)
    response = zeros(2, 2)

    for j in 1:2
        # Column j is the chromaticity response to OSF or OSD.
        step_vector = zeros(2)
        step_vector[j] = knob_step
        response[:, j] = (
            chromaticity_from_tracking(os .+ step_vector; harmonic_strength=harmonic_strength) .-
            chromaticity_from_tracking(os .- step_vector; harmonic_strength=harmonic_strength)
        ) ./ (2knob_step)
    end

    return response
end

function correct_chromaticity(os0; target=[1.0, 1.0], max_iter=5, tolerance=1e-5)
    os = copy(os0)
    history = NamedTuple[]

    for iteration in 1:max_iter
        chromaticity = chromaticity_from_tracking(os)
        residual = target .- chromaticity
        push!(history, (iteration=iteration, os=copy(os), chromaticity=chromaticity))
        norm(residual) < tolerance && break

        # Linear correction: response * dOS = target - current.
        response = chromaticity_response(os)
        os .+= response \ residual
    end

    return os, history
end

initial_os = [2.0, -2.0]
target_chromaticity = [1.0, 1.0]
corrected_os, correction_history = correct_chromaticity(initial_os; target=target_chromaticity)
corrected_chromaticity = chromaticity_from_tracking(corrected_os)
ring_chromaticity_corrected = ring_with_os_knobs(corrected_os)

println("Chromaticity correction history:")
for step in correction_history
    @printf(
        "  iter %d: OSF = % .6f, OSD = % .6f, chrom = (% .6f, % .6f)
",
        step.iteration,
        step.os[1],
        step.os[2],
        step.chromaticity[1],
        step.chromaticity[2],
    )
end

@printf("
Corrected OSF = %.6f
", corrected_os[1])
@printf("Corrected OSD = %.6f
", corrected_os[2])
@printf("Corrected chromaticities = (%.6f, %.6f)
", corrected_chromaticity[1], corrected_chromaticity[2])


LoadError: Unable to get property R_ref from Beamline: Beamline does not have this property

## 12.4 Example: W-Function Optimization

In addition to correcting the linear chromaticity, other optimizations can improve the dynamic aperture. One useful quantity is the W function, which measures the amplitude of the linear energy-dependent beta beat:

$$
W = \sqrt{A^2 + B^2},
$$

where

$$
A = \frac{\partial \alpha}{\partial p_z} - \frac{\alpha}{\beta}\frac{\partial \beta}{\partial p_z}, \qquad
B = \frac{1}{\beta}\frac{\partial \beta}{\partial p_z}.
$$

Here $\beta$ and $\alpha$ are the Twiss functions at the observation point. If $W$ is large, the lattice functions change strongly with momentum offset, which can reduce off-momentum dynamic aperture. Like linear chromaticity, this chromatic beta beat can be corrected with sextupoles. The correction is more delicate because the phase advance between the sextupoles and the quadrupoles that create the W function matters. Since beta beating oscillates with twice the betatron phase advance, sextupoles separated by about $90^\circ$ in betatron phase tend to work against each other. A practical solution is to split the arc sextupoles into separate families, so chromaticity can be corrected while still leaving knobs to reduce the W function.

For the tutorial calculation below, we use a compact 133-element ring rather than the full ESR lattice. The small ring preserves the relative layout needed for this example:

- `ip6` is the reference interaction point;
- `mlrr_6` is just after `ip6`, and `mlrf_6` is just before returning to `ip6`;
- arc 7 follows `ip6`, and arc 5 precedes `ip6`;
- the optimized sextupole knobs are placed in the 5 o'clock and 7 o'clock arcs;
- the phase trombones are placed around the 6 o'clock region, with an `ip4` compensating trombone to keep the net tune shift balanced.

The ideal matching target is to make the W function vanish, so the residual should approach zero in a fully matched example. In this compact ring, the same quantities are calculated and reduced, but the ring was not specially designed to make the W target exactly reachable with this limited knob set. Therefore this section should be read as a SciBmad implementation of the W-function setup and optimization loop, not as the final matched ESR solution.

The varied variables are the arc-5/arc-7 sextupole offsets and the 6 o'clock trombone phase shifts. The optimized quantities are the W-function components in both transverse planes at `ip6` and at `marc_end`, the compact-ring analog of the original `END_7` target.

The complete ESR-ring version is kept separately in `lattices/chapter_12/chapter12_w_optimization.jl`; that is the version to use when trying to reproduce the original near-zero W-function match.


In SciBmad, `twiss(ring; GTPSA_descriptor=Descriptor(6, 2))` returns the Twiss functions as TPSA objects. The coefficient of the first-order $p_z$ term gives $\partial\beta/\partial p_z$ and $\partial\alpha/\partial p_z$, so the W components above can be computed directly from the Twiss table.


In [ ]:
const CH12_TPS_CONSTANT = [0, 0, 0, 0, 0, 0]
const CH12_TPS_PZ = [0, 0, 0, 0, 0, 1]

chapter12_tps_constant(x) = x[CH12_TPS_CONSTANT]
chapter12_tps_pz_coefficient(x) = x[CH12_TPS_PZ]

function chapter12_w_components(beta, alpha)
    beta0 = chapter12_tps_constant(beta)
    alpha0 = chapter12_tps_constant(alpha)
    dbeta_dpz = chapter12_tps_pz_coefficient(beta)
    dalpha_dpz = chapter12_tps_pz_coefficient(alpha)

    # W is the quadrature of these two chromatic-Twiss components.
    A = dalpha_dpz - alpha0 / beta0 * dbeta_dpz
    B = dbeta_dpz / beta0
    W = hypot(A, B)

    return (A=A, B=B, W=W)
end


In [ ]:
# Load the compact 12.4 ring and W-function helper routines.
small_w_script_candidates = [
    joinpath(pwd(), "lattices", "chapter_12", "chapter12_small_ring_w_optimization.jl"),
    joinpath(pwd(), "Ring_Design_Tutorial_SciBmad", "lattices", "chapter_12", "chapter12_small_ring_w_optimization.jl"),
]
small_w_script = first(filter(isfile, small_w_script_candidates))
include(small_w_script)

@printf("Small 12.4 ring elements: %d
", length(small_ring_12_4.line))


In [ ]:
# First compute W explicitly at ip6 so the definition above is visible in use.
apply_small_w_knobs!(zeros(8))
initial_w_twiss = twiss(small_ring_12_4; GTPSA_descriptor=Descriptor(6, 2))
ip6_index = SMALL_W_TARGET_INDICES[1]

ip6_horizontal_w = chapter12_w_components(
    initial_w_twiss.table.beta_1[ip6_index],
    initial_w_twiss.table.alpha_1[ip6_index],
)
ip6_vertical_w = chapter12_w_components(
    initial_w_twiss.table.beta_2[ip6_index],
    initial_w_twiss.table.alpha_2[ip6_index],
)

@printf("Explicit W at ip6 before optimization:
")
@printf("  horizontal: A = % .6g, B = % .6g, W = % .6g
", ip6_horizontal_w.A, ip6_horizontal_w.B, ip6_horizontal_w.W)
@printf("  vertical:   A = % .6g, B = % .6g, W = % .6g
", ip6_vertical_w.A, ip6_vertical_w.B, ip6_vertical_w.W)

# Then report the full residual vector and run a short example optimization.
# The returned vector contains [SF5, SD5, SF7, SD7, dnu_x_left, dnu_x_right, dnu_y_left, dnu_y_right].
println("
Initial small-ring W report:")
small_w_report(zeros(8))

optimized_small_w_knobs = optimize_small_w_function(n_iterations=6)
